# Pemulihan Stempel Waktu — Transkrip Acuan Jalur wav2vec2
### Pengenalan Keilmuan Komputer · Rekayasa Perangkat Lunak · Jaringan Nirkabel

Transkrip acuan untuk tiga mata kuliah ini disusun melalui jalur yang berbeda: rekaman disegmentasi lebih dahulu dengan Silero VAD, lalu tiap segmen ditranskripsi dan dikoreksi. Manifest yang dihasilkan hanya menyimpan **durasi** tiap segmen, tanpa stempel waktu absolut terhadap rekaman.

Notebook ini memulihkan stempel waktu itu, lalu menyatukan hasilnya dengan master transkrip acuan jalur Whisper.

## Alur

```
[1] Pemetaan rekaman          -> file_id, mata kuliah, URL
[2] Pemulihan stempel waktu   -> jalankan ulang VAD, simpan offset
[3] Verifikasi                -> durasi hasil == durasi manifest
[4] Master TSV                -> data/output/master/<ID>.tsv
[5] Praproses audio           -> data/processed/audio/  16 kHz mono, loudnorm
[6] Baris manifest            -> digabung ke manifest_rekaman.csv
[7] Laporan cakupan           -> untuk Bab 4
```

## Yang perlu dicatat di Bab 3

Dua hal berbeda dari jalur Whisper, dan keduanya perlu dinyatakan:

1. **Cakupan transkrip bukan seluruh durasi rekaman.** Segmentasi memakai `min_speech_duration_ms` 1000 dan ambang segmen minimum 3 detik, sehingga ujaran yang lebih pendek dari itu tidak masuk transkrip. Penilaian untuk rekaman ini dibatasi pada rentang waktu yang tercakup.
2. **Delapan rekaman terpotong di sekitar menit ke-19** pada tahap pengunduhan. Durasi setelah pemotongan itulah yang dipakai sebagai penyebut RTF dan dilaporkan sebagai durasi rekaman.

## 0. Persiapan

In [ ]:
# Jalankan sekali bila pustaka belum terpasang.
# !pip install silero-vad torch librosa soundfile pandas
# FFmpeg harus tersedia di PATH.


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from pathlib import Path
import json, subprocess, shutil, datetime
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW        = ROOT / "data" / "raw"
PROC       = ROOT / "data" / "processed"
OUT        = ROOT / "data" / "output"

RAW_SRC    = RAW / "audio_src_caca" / "raw_finetune"
SEG_SRC    = RAW / "audio_src_segments_caca" / "segments"
CSV_SRC    = OUT / "transkrip_acuan_caca.csv"

AUDIO_PROC = PROC / "audio"
MASTER     = OUT  / "master"
TS_DIR     = OUT  / "stempel_waktu"

for d in (AUDIO_PROC, MASTER, TS_DIR, OUT):
    d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUT / "manifest_rekaman.csv"

SEG = {
    "sample_rate": 16000,
    "min_speech_duration_ms": 1000,
    "speech_pad_ms": 300,
    "min_silence_duration_ms": 500,
    "min_seg_duration": 3.0,
    "max_seg_duration": 20.0,
}

PRA = {
    "sample_rate": 16000, "channels": 1,
    "loudnorm_I": -23.0, "loudnorm_TP": -2.0, "loudnorm_LRA": 7.0,
}

print("Akar proyek :", ROOT)
print("FFmpeg      :", shutil.which("ffmpeg") or "TIDAK DITEMUKAN")
print("WAV sumber  :", RAW_SRC, "|", "ADA" if RAW_SRC.exists() else "TIDAK ADA")
print("Manifest    :", CSV_SRC, "|", "ADA" if CSV_SRC.exists() else "TIDAK ADA")

Akar proyek : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative
FFmpeg      : C:\Users\Aufii\AppData\Local\Microsoft\WinGet\Links\ffmpeg.EXE
WAV sumber  : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\raw\audio_src_caca\raw_finetune | ADA
Manifest    : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\output\transkrip_acuan_caca.csv | ADA


## 1. Pemetaan rekaman

Nama berkas pada jalur wav2vec2 (`pkk1`, `rpl2`, `jarnir3`) dipetakan ke `file_id` bergaya sama dengan jalur Whisper, agar kedua master dapat digabung.

Kolom `durasi_youtube_detik` disimpan terpisah dari durasi berkas. Selisih keduanya adalah bagian rekaman yang tidak pernah terunduh, dan itu yang perlu dilaporkan.

In [15]:
PETA = [
    ("pkk1",    "PKK-01",    "PKK",    "itY08MIbkXE", "19:41"),
    ("pkk2",    "PKK-02",    "PKK",    "IGV0qIrsaTA", "16:52"),
    ("pkk3",    "PKK-03",    "PKK",    "4BF7mKzfN50", "10:44"),
    ("pkk4",    "PKK-04",    "PKK",    "9O8NHGYmAd0", "10:28"),
    ("pkk5",    "PKK-05",    "PKK",    "3wR5xW_rzNE", "23:28"),

    ("rpl1",    "RPL-01",    "RPL",    "7Q_JLyUJulI", "17:28"),
    ("rpl2",    "RPL-02",    "RPL",    "66VCV_RUV9Q", "21:50"),
    ("rpl3",    "RPL-03",    "RPL",    "JI74EecGFQg", "24:15"),
    ("rpl4",    "RPL-04",    "RPL",    "wWGTCzryEVc", "27:38"),
    ("rpl5",    "RPL-05",    "RPL",    "wEbIgmcmn3Y", "24:04"),
    ("rpl6",    "RPL-06",    "RPL",    "lwotPCtoP4c", "18:17"),
    ("rpl7",    "RPL-07",    "RPL",    "t8DRqvwG4UU", "20:14"),
    ("rpl8",    "RPL-08",    "RPL",    "6OU9phN5dDQ", "11:08"),
    ("rpl9",    "RPL-09",    "RPL",    "XzSYCB2gTiw", "14:40"),
    ("rpl10",   "RPL-10",    "RPL",    "Ue0soxUargI", "16:48"),
    ("rpl11",   "RPL-11",    "RPL",    "rCnYFe6YJjM", "28:30"),

    ("jarnir1", "JARNIR-01", "JARNIR", "eX3xCGoV8dg", "26:03"),
    ("jarnir2", "JARNIR-02", "JARNIR", "Neglyg4POco", "21:56"),
    ("jarnir3", "JARNIR-03", "JARNIR", "OPE6XYo-Y_M", "26:56"),
    ("jarnir4", "JARNIR-04", "JARNIR", "DDi-ec2OHJg", "30:56"),
    ("jarnir5", "JARNIR-05", "JARNIR", "vi40zQkBs7M", "14:42"),
]

NAMA_MK = {
    "PKK":    "Pengenalan Keilmuan Komputer",
    "RPL":    "Rekayasa Perangkat Lunak",
    "JARNIR": "Jaringan Nirkabel",
}

DUPLIKAT = {"RPL-08": "beririsan dengan APS-02 (video yang sama); keluarkan salah satu"}

KELUARKAN = {
    "RPL-08": "rekaman sama dengan APS-02; APS-02 yang dipertahankan",
}

def mmss(t):
    b = [int(x) for x in t.split(":")]
    return b[0]*60 + b[1] if len(b) == 2 else b[0]*3600 + b[1]*60 + b[2]

peta = pd.DataFrame(
    [{"sumber": s, "file_id": f, "mata_kuliah": mk,
      "url": f"https://www.youtube.com/watch?v={yid}",
      "durasi_tertera": d, "durasi_youtube_detik": mmss(d)}
     for s, f, mk, yid, d in PETA])

csv_src = pd.read_csv(CSV_SRC, keep_default_na=False)
csv_src["duration_sec"] = csv_src["duration_sec"].astype(float)
ada_csv = set(csv_src["source_video"])
ada_wav = {p.stem for p in RAW_SRC.glob("*.[wW][aA][vV]")}

peta["ada_manifest"] = peta["sumber"].isin(ada_csv)
peta["ada_wav"] = peta["sumber"].isin(ada_wav)
peta["dikeluarkan"] = peta["file_id"].isin(KELUARKAN)
peta["siap"] = peta["ada_manifest"] & peta["ada_wav"] & ~peta["dikeluarkan"]

print(peta[["sumber", "file_id", "mata_kuliah", "durasi_tertera",
            "ada_manifest", "ada_wav", "siap"]].to_string(index=False))

if peta["dikeluarkan"].any():
    print("\nDikeluarkan:")
    for _, r in peta[peta["dikeluarkan"]].iterrows():
        print(f"  {r['file_id']}  ({r['sumber']})  — {KELUARKAN[r['file_id']]}")
        
siap = peta[peta["siap"]]
print(f"\nSiap diproses: {len(siap)} dari {len(peta)} rekaman")
belum = peta[~peta["siap"]]
if len(belum):
    print("Belum tersedia:", ", ".join(belum["sumber"]))

 sumber   file_id mata_kuliah durasi_tertera  ada_manifest  ada_wav  siap
   pkk1    PKK-01         PKK          19:41          True     True  True
   pkk2    PKK-02         PKK          16:52          True     True  True
   pkk3    PKK-03         PKK          10:44          True     True  True
   pkk4    PKK-04         PKK          10:28         False    False False
   pkk5    PKK-05         PKK          23:28          True     True  True
   rpl1    RPL-01         RPL          17:28         False    False False
   rpl2    RPL-02         RPL          21:50          True     True  True
   rpl3    RPL-03         RPL          24:15          True     True  True
   rpl4    RPL-04         RPL          27:38          True     True  True
   rpl5    RPL-05         RPL          24:04         False    False False
   rpl6    RPL-06         RPL          18:17          True     True  True
   rpl7    RPL-07         RPL          20:14          True     True  True
   rpl8    RPL-08         RPL         

## 2. Pemulihan stempel waktu

Langkah di bawah menyalin persis urutan operasi skrip segmentasi aslinya: rerata kanal, pengubahan laju cuplik ke 16 kHz, normalisasi puncak ke 0,95, lalu Silero VAD dengan parameter yang sama. Bedanya hanya satu — `start` dan `end` ikut disimpan.

Urutan operasi harus dipertahankan. Menukar normalisasi dengan pengubahan laju cuplik, atau mengganti normalisasi puncak dengan normalisasi lantang, akan menggeser batas segmen dan membuat pemulihan gagal.

In [16]:
import torch, soundfile as sf, librosa
from silero_vad import load_silero_vad, get_speech_timestamps

vad_model = load_silero_vad()
print("Silero VAD dimuat.")


def muat_seperti_asli(path):
    "Rerata kanal, ubah laju cuplik, normalisasi puncak — urutan asli."
    x, sr = sf.read(str(path))
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    if sr != SEG["sample_rate"]:
        x = librosa.resample(x, orig_sr=sr, target_sr=SEG["sample_rate"])
    puncak = np.max(np.abs(x))
    if puncak > 0:
        x = x / puncak * 0.95
    return x


def pulihkan(sumber):
    "Kembalikan daftar segmen dengan start/end dalam detik."
    x = muat_seperti_asli(RAW_SRC / f"{sumber}.wav")
    sr = SEG["sample_rate"]

    ts = get_speech_timestamps(
        torch.from_numpy(x).float(), vad_model, sampling_rate=sr,
        min_speech_duration_ms=SEG["min_speech_duration_ms"],
        speech_pad_ms=SEG["speech_pad_ms"],
        min_silence_duration_ms=SEG["min_silence_duration_ms"],
    )

    maks = int(SEG["max_seg_duration"] * sr)
    hasil = []
    for t in ts:
        a, b = t["start"], t["end"]
        if (b - a) / sr > SEG["max_seg_duration"]:
            potong, pos = [], a
            while pos < b:
                e = min(pos + maks, b)
                potong.append((pos, e))
                pos = e
        else:
            potong = [(a, b)]
        for p, q in potong:
            d = (q - p) / sr
            if d < SEG["min_seg_duration"]:
                continue
            hasil.append({"start": p / sr, "end": q / sr, "dur": d})
    return hasil, len(x) / sr


pulih = {}
for _, r in siap.iterrows():
    seg, durasi_wav = pulihkan(r["sumber"])
    pulih[r["sumber"]] = {"segmen": seg, "durasi_wav": durasi_wav}
    print(f"{r['sumber']:10s} {len(seg):4d} segmen  "
          f"wicara {sum(s['dur'] for s in seg):7.1f}s / wav {durasi_wav:7.1f}s")

Silero VAD dimuat.
pkk1        127 segmen  wicara   746.0s / wav  1181.0s
pkk2        116 segmen  wicara   809.4s / wav  1012.0s
pkk3         62 segmen  wicara   565.4s / wav   644.0s
pkk5        119 segmen  wicara  1047.2s / wav  1146.0s
rpl2        117 segmen  wicara  1082.2s / wav  1140.0s
rpl3        113 segmen  wicara  1092.5s / wav  1140.0s
rpl4        110 segmen  wicara  1083.9s / wav  1140.0s
rpl6        119 segmen  wicara  1049.4s / wav  1097.0s
rpl7        131 segmen  wicara  1172.5s / wav  1214.0s
rpl9        102 segmen  wicara   833.7s / wav   880.0s
rpl10       103 segmen  wicara   954.7s / wav  1008.0s
rpl11       111 segmen  wicara  1030.8s / wav  1130.0s
jarnir1     123 segmen  wicara   926.6s / wav  1140.0s
jarnir3     106 segmen  wicara  1002.2s / wav  1090.0s
jarnir4     103 segmen  wicara  1056.7s / wav  1140.0s
jarnir5      50 segmen  wicara   853.3s / wav   882.0s


## 3. Verifikasi

Durasi hasil pemulihan dibandingkan dengan manifest asli, segmen demi segmen, pada ketelitian dua angka desimal — sama dengan pembulatan yang dipakai manifest.

Rekaman yang tidak lolos tidak diproses lebih lanjut. Lebih baik kehilangan satu rekaman daripada memasang stempel waktu yang salah pada transkripnya.

In [17]:
lap = []
lolos = []
for _, r in siap.iterrows():
    s = r["sumber"]
    hasil = [round(x["dur"], 2) for x in pulih[s]["segmen"]]
    asli  = csv_src.loc[csv_src["source_video"] == s, "duration_sec"].round(2).tolist()

    cocok = hasil == asli
    n_sama = sum(1 for a, b in zip(hasil, asli) if a == b)
    lap.append({"sumber": s, "file_id": r["file_id"],
                "n_pulih": len(hasil), "n_manifest": len(asli),
                "n_cocok": n_sama, "identik": cocok})
    if cocok:
        lolos.append(s)

df_lap = pd.DataFrame(lap)
print(df_lap.to_string(index=False))

n_ok = int(df_lap["identik"].sum())
print(f"\n{n_ok} dari {len(df_lap)} rekaman identik.")
if n_ok < len(df_lap):
    print("\nTIDAK LOLOS:")
    print(df_lap[~df_lap["identik"]].to_string(index=False))
    print("\nRekaman tersebut dilewati. Periksa apakah berkas WAV-nya sama")
    print("dengan yang dipakai pada segmentasi asli.")
assert n_ok > 0, "Tidak ada rekaman yang lolos verifikasi; pemulihan dibatalkan."

 sumber   file_id  n_pulih  n_manifest  n_cocok  identik
   pkk1    PKK-01      127         127      127     True
   pkk2    PKK-02      116         116      116     True
   pkk3    PKK-03       62          62       62     True
   pkk5    PKK-05      119         119      119     True
   rpl2    RPL-02      117         117      117     True
   rpl3    RPL-03      113         113      113     True
   rpl4    RPL-04      110         110      110     True
   rpl6    RPL-06      119         119      119     True
   rpl7    RPL-07      131         131      131     True
   rpl9    RPL-09      102         102      102     True
  rpl10    RPL-10      103         103      103     True
  rpl11    RPL-11      111         111      111     True
jarnir1 JARNIR-01      123         123      123     True
jarnir3 JARNIR-03      106         106      106     True
jarnir4 JARNIR-04      103         103      103     True
jarnir5 JARNIR-05       50          50       50     True

16 dari 16 rekaman identik.


## 4. Master TSV

Skemanya sama dengan master jalur Whisper, sehingga keduanya dapat disatukan.

Dua kolom berbeda isinya:

- `teks_asr` dikosongkan. Jalur ini tidak menyimpan arsip keluaran mentah, sehingga tidak ada pembanding untuk menghitung proporsi koreksi.
- `flag` diisi `jalur-wav2vec2` pada seluruh baris, sebagai penanda bahwa transkrip ini melewati normalisasi di hulu: huruf kecil, tanpa tanda baca, tanpa tanda hubung. Tahap normalisasi 3.6.3 perlu memperlakukannya berbeda.

In [18]:
KOLOM = ["file_id", "mata_kuliah", "penutur", "seg_id", "start", "end", "dur",
         "teks_asr", "teks_acuan", "flag", "catatan"]

semua = []
for _, r in siap.iterrows():
    s = r["sumber"]
    if s not in lolos:
        continue

    seg = pulih[s]["segmen"]
    teks = csv_src.loc[csv_src["source_video"] == s, "text"].tolist()
    assert len(seg) == len(teks)

    baris = [{
        "file_id": r["file_id"],
        "mata_kuliah": r["mata_kuliah"],
        "penutur": "",
        "seg_id": i,
        "start": f"{x['start']:.3f}",
        "end":   f"{x['end']:.3f}",
        "dur":   f"{x['dur']:.3f}",
        "teks_asr": "",
        "teks_acuan": str(t).strip(),
        "flag": "jalur-wav2vec2",
        "catatan": (""),
    } for i, (x, t) in enumerate(zip(seg, teks), start=1)]

    df = pd.DataFrame(baris, columns=KOLOM)
    df.to_csv(MASTER / f"{r['file_id']}.tsv", sep="\t", index=False, encoding="utf-8-sig")
    semua.append(df)

    # arsip stempel waktu, untuk jejak audit
    (TS_DIR / f"{r['file_id']}.json").write_text(json.dumps({
        "file_id": r["file_id"], "sumber": s,
        "durasi_wav": round(pulih[s]["durasi_wav"], 3),
        "parameter_segmentasi": SEG,
        "segmen": [{"seg_id": i, "start": round(x["start"], 3), "end": round(x["end"], 3)}
                   for i, x in enumerate(seg, start=1)],
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"{r['file_id']:10s} {len(df):4d} segmen")

master_caca = pd.concat(semua, ignore_index=True) if semua else None
if master_caca is not None:
    master_caca.to_csv(OUT / "transkrip_acuan_wav2vec2.tsv", sep="\t",
                       index=False, encoding="utf-8-sig")
    print(f"\nGabungan: {len(master_caca)} segmen dari "
          f"{master_caca['file_id'].nunique()} rekaman")

PKK-01      127 segmen
PKK-02      116 segmen
PKK-03       62 segmen
PKK-05      119 segmen
RPL-02      117 segmen
RPL-03      113 segmen
RPL-04      110 segmen
RPL-06      119 segmen
RPL-07      131 segmen
RPL-09      102 segmen
RPL-10      103 segmen
RPL-11      111 segmen
JARNIR-01   123 segmen
JARNIR-03   106 segmen
JARNIR-04   103 segmen
JARNIR-05    50 segmen

Gabungan: 1712 segmen dari 16 rekaman


## 5. Praproses audio

Rekaman diproses dengan parameter yang sama seperti jalur Whisper: 16 kHz, mono, PCM 16-bit, normalisasi lantang EBU R128. Hasilnya diletakkan bersama rekaman jalur Whisper agar Tahap 1 sampai Tahap 3 berjalan atas audio yang seragam.

Berkas segmen dari jalur wav2vec2 **tidak** dipakai. Segmen itu dinormalisasi puncak, bukan lantang; mencampurnya dengan rekaman yang dinormalisasi lantang akan membuat audio latih tidak homogen. Yang diambil dari jalur itu hanya teks dan stempel waktunya.

In [19]:
def durasi_detik(berkas):
    h = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(berkas)],
        capture_output=True, text=True)
    try:
        return float(h.stdout.strip())
    except ValueError:
        return None


def praproses(sumber, file_id, force=False):
    asal = RAW_SRC / f"{sumber}.wav"
    tuju = AUDIO_PROC / f"{file_id}.wav"
    if tuju.exists() and not force:
        return durasi_detik(tuju), "sudah ada"
    h = subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(asal),
        "-af", f"loudnorm=I={PRA['loudnorm_I']}:TP={PRA['loudnorm_TP']}:LRA={PRA['loudnorm_LRA']}",
        "-ar", str(PRA["sample_rate"]), "-ac", str(PRA["channels"]),
        "-c:a", "pcm_s16le", "-map_metadata", "-1", str(tuju),
    ], capture_output=True, text=True)
    if h.returncode != 0:
        return None, "GAGAL: " + h.stderr.strip()[:120]
    return durasi_detik(tuju), "diproses"


FORCE_PRAPROSES = False

durasi_proc = {}
for _, r in siap.iterrows():
    if r["sumber"] not in lolos:
        continue
    d, status = praproses(r["sumber"], r["file_id"], FORCE_PRAPROSES)
    durasi_proc[r["file_id"]] = d
    print(f"{r['file_id']:10s} {status:12s} {d if d is None else f'{d:8.1f}s'}")

PKK-01     sudah ada      1181.0s
PKK-02     sudah ada      1012.0s
PKK-03     sudah ada       644.0s
PKK-05     sudah ada      1146.0s
RPL-02     sudah ada      1140.0s
RPL-03     sudah ada      1140.0s
RPL-04     sudah ada      1140.0s
RPL-06     sudah ada      1097.0s
RPL-07     sudah ada      1214.0s
RPL-09     sudah ada       880.0s
RPL-10     sudah ada      1008.0s
RPL-11     sudah ada      1130.0s
JARNIR-01  sudah ada      1140.0s
JARNIR-03  sudah ada      1090.0s
JARNIR-04  sudah ada      1140.0s
JARNIR-05  sudah ada       882.0s


## 6. Baris manifest

Baris untuk rekaman jalur wav2vec2 ditambahkan ke `manifest_rekaman.csv` yang sudah ada, tanpa menimpa baris jalur Whisper.

Tiga kolom tambahan dibutuhkan jalur ini dan akan muncul kosong pada baris jalur Whisper:

| Kolom | Isi |
|---|---|
| `durasi_youtube_detik` | durasi tertera di sumber |
| `cakupan_acuan` | durasi bertranskrip dibagi durasi berkas |
| `jalur_acuan` | `whisper` atau `wav2vec2` |

Kolom `ada_takarir_manusia` dan `lolos_penyaringan` dikosongkan — keduanya tetap perlu diisi manual, sama seperti jalur Whisper.

In [20]:
KOLOM_MANIFEST = [
    "file_id", "mata_kuliah", "nama_mata_kuliah", "url", "penutur",
    "durasi_tertera", "durasi_tertera_detik",
    "trim_awal_detik", "trim_akhir_detik",
    "durasi_asli_detik", "durasi_potong_detik",
    "durasi_youtube_detik", "cakupan_acuan", "jalur_acuan",
    "ada_takarir_manusia", "lolos_penyaringan", "split",
    "tanggal_unduh", "catatan",
]

baris_baru = []
for _, r in siap.iterrows():
    s, fid = r["sumber"], r["file_id"]
    if s not in lolos:
        continue
    d_wav = pulih[s]["durasi_wav"]
    d_wicara = sum(x["dur"] for x in pulih[s]["segmen"])
    terpotong = r["durasi_youtube_detik"] - d_wav

    cat = []
    if terpotong > 2:
        cat.append(f"terpotong {terpotong:.0f} detik pada tahap pengunduhan")

    baris_baru.append({
        "file_id": fid,
        "mata_kuliah": r["mata_kuliah"],
        "nama_mata_kuliah": NAMA_MK[r["mata_kuliah"]],
        "url": r["url"],
        "penutur": "",
        "durasi_tertera": r["durasi_tertera"],
        "durasi_tertera_detik": f"{d_wav:.3f}",
        "trim_awal_detik": "0",
        "trim_akhir_detik": "0",
        "durasi_asli_detik": f"{d_wav:.3f}",
        "durasi_potong_detik": f"{durasi_proc.get(fid) or d_wav:.3f}",
        "durasi_youtube_detik": f"{r['durasi_youtube_detik']:.3f}",
        "cakupan_acuan": f"{d_wicara / d_wav:.3f}",
        "jalur_acuan": "wav2vec2",
        "ada_takarir_manusia": "",
        "lolos_penyaringan": "",
        "split": "",
        "tanggal_unduh": "",
        "catatan": "; ".join(cat),
    })

df_baru = pd.DataFrame(baris_baru, columns=KOLOM_MANIFEST)

if MANIFEST_PATH.exists():
    lama = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False)
    for k in KOLOM_MANIFEST:
        if k not in lama.columns:
            lama[k] = ""
    lama.loc[lama["jalur_acuan"] == "", "jalur_acuan"] = "whisper"
    gab = pd.concat([lama, df_baru[~df_baru["file_id"].isin(lama["file_id"])]],
                    ignore_index=True)[KOLOM_MANIFEST]
else:
    gab = df_baru

gab.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")
print(f"Manifest: {len(gab)} rekaman")
print(gab.groupby(["jalur_acuan", "mata_kuliah"]).size().to_string())
print()
print(df_baru[["file_id", "durasi_asli_detik", "durasi_youtube_detik",
               "cakupan_acuan", "catatan"]].to_string(index=False))

Manifest: 35 rekaman
jalur_acuan  mata_kuliah
wav2vec2     JARNIR         4
             PKK            4
             RPL            8
whisper      AOK            8
             APS            6
             IMK            5

  file_id durasi_asli_detik durasi_youtube_detik cakupan_acuan                                    catatan
   PKK-01          1181.014             1181.000         0.632                                           
   PKK-02          1012.014             1012.000         0.800                                           
   PKK-03           644.014              644.000         0.878                                           
   PKK-05          1145.994             1408.000         0.914 terpotong 262 detik pada tahap pengunduhan
   RPL-02          1140.014             1310.000         0.949 terpotong 170 detik pada tahap pengunduhan
   RPL-03          1140.014             1455.000         0.958 terpotong 315 detik pada tahap pengunduhan
   RPL-04          1140.014    

## 7. Laporan cakupan

Dua angka yang perlu masuk Bab 4.

**Cakupan acuan** — berapa bagian rekaman yang punya transkrip. Rentang di luar itu tidak dinilai, sebab tidak ada acuannya. Nilai di bawah 0,85 perlu disebut terpisah.

**Durasi yang tidak terunduh** — selisih antara durasi di sumber dan durasi berkas. Ini bukan keputusan metodologis melainkan keterbatasan teknis, dan perlu dinyatakan apa adanya.

In [21]:
if master_caca is not None:
    lap = df_baru.copy()
    for k in ("durasi_asli_detik", "durasi_youtube_detik", "cakupan_acuan"):
        lap[k] = lap[k].astype(float)
    lap["tak_terunduh"] = (lap["durasi_youtube_detik"] - lap["durasi_asli_detik"]).round(1)

    kata = (master_caca.assign(n=master_caca["teks_acuan"].str.split().str.len())
                       .groupby("file_id")["n"].sum())
    lap = lap.merge(kata.rename("jumlah_kata"), left_on="file_id", right_index=True)

    print(lap[["file_id", "durasi_asli_detik", "durasi_youtube_detik",
               "tak_terunduh", "cakupan_acuan", "jumlah_kata"]].to_string(index=False))

    print(f"\nCakupan: rerata {lap.cakupan_acuan.mean():.3f} | "
          f"min {lap.cakupan_acuan.min():.3f} ({lap.loc[lap.cakupan_acuan.idxmin(),'file_id']}) | "
          f"maks {lap.cakupan_acuan.max():.3f}")

    rendah = lap[lap.cakupan_acuan < 0.85]
    if len(rendah):
        print("\nCakupan di bawah 0,85 — sebut terpisah di Bab 4:")
        print(rendah[["file_id", "cakupan_acuan"]].to_string(index=False))

    potong = lap[lap.tak_terunduh > 2]
    if len(potong):
        print(f"\nTerpotong saat pengunduhan: {len(potong)} rekaman, "
              f"total {potong.tak_terunduh.sum():.0f} detik "
              f"({potong.tak_terunduh.sum()/60:.0f} menit)")

    ring = (lap.groupby("mata_kuliah")
               .agg(rekaman=("file_id", "count"),
                    durasi_detik=("durasi_asli_detik", "sum"),
                    cakupan_rerata=("cakupan_acuan", "mean"),
                    kata=("jumlah_kata", "sum")))
    ring["durasi_jam"] = (ring.durasi_detik / 3600).round(2)
    ring["cakupan_rerata"] = ring.cakupan_rerata.round(3)
    print("\nPer mata kuliah:")
    print(ring[["rekaman", "durasi_jam", "cakupan_rerata", "kata"]].to_string())

    lap.to_csv(OUT / "cakupan_wav2vec2.csv", index=False, encoding="utf-8-sig")

  file_id  durasi_asli_detik  durasi_youtube_detik  tak_terunduh  cakupan_acuan  jumlah_kata
   PKK-01           1181.014                1181.0          -0.0          0.632         1236
   PKK-02           1012.014                1012.0          -0.0          0.800         1156
   PKK-03            644.014                 644.0          -0.0          0.878         1020
   PKK-05           1145.994                1408.0         262.0          0.914         1919
   RPL-02           1140.014                1310.0         170.0          0.949         2268
   RPL-03           1140.014                1455.0         315.0          0.958         2489
   RPL-04           1140.014                1658.0         518.0          0.951         2592
   RPL-06           1097.014                1097.0          -0.0          0.957         2445
   RPL-07           1214.014                1214.0          -0.0          0.966         2787
   RPL-09            880.014                 880.0          -0.0      

In [22]:
# Metadata run.
import platform, sys
meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "pemulihan stempel waktu jalur wav2vec2",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "parameter_segmentasi": SEG,
    "parameter_praproses": PRA,
    "rekaman_lolos": sorted(lolos),
    "versi_pustaka": {},
}
for nama in ["torch", "librosa", "soundfile", "numpy", "pandas", "onnxruntime"]:
    try:
        meta["versi_pustaka"][nama] = __import__(nama).__version__
    except Exception:
        meta["versi_pustaka"][nama] = None

(OUT / "run_meta_wav2vec2.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta["versi_pustaka"], indent=2))

{
  "torch": "2.4.1+cu121",
  "librosa": "0.10.2.post1",
  "soundfile": "0.13.1",
  "numpy": "2.0.2",
  "pandas": "3.0.3",
  "onnxruntime": "1.26.0"
}


---

## Keluaran

| Berkas | Isi |
|---|---|
| `data/output/master/<ID>.tsv` | master transkrip acuan per rekaman, dengan stempel waktu |
| `data/output/transkrip_acuan_wav2vec2.tsv` | master gabungan jalur ini |
| `data/output/stempel_waktu/<ID>.json` | arsip batas segmen hasil pemulihan |
| `data/processed/audio/<ID>.wav` | 16 kHz mono, loudnorm, siap untuk Tahap 1–3 |
| `data/output/manifest_rekaman.csv` | manifest gabungan kedua jalur |
| `data/output/cakupan_wav2vec2.csv` | tabel cakupan untuk Bab 4 |
| `data/output/run_meta_wav2vec2.json` | parameter dan versi pustaka |

## Yang masih perlu diisi manual

`ada_takarir_manusia` dan `lolos_penyaringan` untuk rekaman jalur ini, sama seperti jalur Whisper.

## Yang perlu diputuskan

**RPL-08 dan APS-02 adalah rekaman yang sama.** Salah satu harus dikeluarkan sebelum pembagian data 3.6.2. Membiarkan keduanya berarti rekaman yang sama berpeluang masuk himpunan latih dan himpunan uji sekaligus, dan hasil Tahap 3 tidak lagi sah.

## Penilaian terbatas cakupan

Rekaman jalur ini punya transkrip hanya pada rentang waktu tertentu. Pada Tahap 1 sampai Tahap 3, inferensi tetap dijalankan atas **rekaman utuh** — RTF membutuhkan penyebut berupa durasi rekaman penuh (3.6.8), dan Tahap 2 membutuhkan jendela yang bergulir sepanjang rekaman (3.6.6).

Yang dibatasi adalah penilaiannya: untuk tiap segmen acuan, teks hipotesis yang dibandingkan hanyalah yang beririsan waktu dengan segmen itu. Rentang tanpa acuan tidak menyumbang operasi sunting apa pun. Pendekatan ini sama dengan yang dipakai mengisi kolom `teks_asr` pada jalur Whisper.

Konsekuensinya perlu dinyatakan di Bab 4: perbandingan WER **antarmata kuliah** harus dibaca hati-hati karena cakupannya berbeda, sedangkan perbandingan **antarkonfigurasi** tetap sah karena setiap konfigurasi menghadapi cakupan yang sama persis.

## Perbedaan normalisasi

Transkrip jalur ini sudah dinormalisasi di hulu: huruf kecil, tanpa tanda baca, tanpa tanda hubung, tanpa akronim berhuruf kapital. Transkrip jalur Whisper verbatim.

Praproses teks 3.6.3 menerapkan prosedur yang sama pada acuan dan hipotesis, dan sebagian besar perbedaan itu hilang setelahnya — huruf kecil dan tanda baca sama-sama dibuang. Yang tidak hilang adalah **tanda hubung pada bentuk ulang**: `kata-kata` dipertahankan sebagai satu kata oleh Tabel 3.4, sedangkan pada jalur ini sudah menjadi dua kata secara permanen. Itu menggeser penyebut WER dan CER untuk rekaman jalur ini.

Kolom `flag` bernilai `jalur-wav2vec2` menandai baris yang terdampak, sehingga tahap normalisasi dapat memperlakukannya terpisah bila diperlukan.